In [15]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import BaseTool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel
from typing import TypedDict
from dotenv import load_dotenv
import time
import asyncio
import threading

In [2]:
load_dotenv()

True

In [3]:
# Dedicated async loop for backend tasks
_ASYNC_LOOP = asyncio.new_event_loop()
_ASYNC_THREAD = threading.Thread(target=_ASYNC_LOOP.run_forever, daemon=True)
_ASYNC_THREAD.start()

In [5]:
llm = ChatOpenAI()

In [4]:
def _submit_async(coro):
    return asyncio.run_coroutine_threadsafe(coro, _ASYNC_LOOP)

def run_async(coro):
    return _submit_async(coro).result()

def submit_async_task(coro):
    """Schedule a coroutine on the backend event loop."""
    return _submit_async(coro)

In [24]:
client = MultiServerMCPClient(
    {
        "tripplanning": {
            "transport": "stdio",
            "command": "python3",
            "args": ["C:\\Users\\g.chetan.bhardwaj\\Documents\\Learn\\trip_planning_chatbot\\mcp\\main.py"]
        }
    }
)

In [22]:
search_tool = DuckDuckGoSearchRun()

def load_mcp_tools() -> list[BaseTool]:
    try:
        return run_async(client.get_tools())
    except Exception:
        return []

mcp_tools = load_mcp_tools()

tools = [search_tool, *mcp_tools]

In [ ]:
print(tools)

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'))]


In [11]:
llm_with_tools = llm.bind_tools(tools) if tools else llm

In [18]:
def chat_node(state:MessagesState):
        """Invokes LLM to get a response based on user input"""

        messages = state['messages']
        response = llm_with_tools.invoke(messages)

        return {'messages': [response]}

tool_node = ToolNode(tools) if tools else None

builder = StateGraph(MessagesState)

builder.add_node('chat_node', chat_node)
builder.add_edge(START, 'chat_node')

if tool_node:
    builder.add_node('tools', tool_node)
    builder.add_conditional_edges('chat_node', tools_condition)
    builder.add_edge('tools', 'chat_node')
else:
    builder.add_edge('chat_node', END)

checkpointer = InMemorySaver()

chatbot = builder.compile(checkpointer=checkpointer)

In [19]:
config = {"configurable": {"thread_id": "thread-1"}}

response = chatbot.invoke({'messages': HumanMessage(content="What hotels are there for mumbai?")}, config=config)
print(response['messages'][-1].content)

could not parse an IP from hosts file ("Atul")
could not parse an IP from hosts file ("Atul")
could not parse an IP from hosts file ("10.148.220.1�����")
could not parse an IP from hosts file ("10.148.220.1�����")
could not parse an IP from hosts file ("10.148.220.167�")
could not parse an IP from hosts file ("10.148.220.167�")
could not parse an IP from hosts file ("10.148.220.179�")
could not parse an IP from hosts file ("10.148.220.179�")
could not parse an IP from hosts file ("Atul")
could not parse an IP from hosts file ("10.148.220.1�����")
could not parse an IP from hosts file ("10.148.220.167�")
could not parse an IP from hosts file ("10.148.220.179�")


Here are some notable hotels in Mumbai:

1. Taj Mahal Tower, Mumbai
2. Trident Nariman Point
3. The Taj Mahal Palace, Mumbai
4. The St. Regis Mumbai
5. The Oberoi Mumbai

These hotels are located in South Mumbai and offer various luxurious accommodations for your stay.
